# calo-ddpm — Combined Statistical Analysis & Shrinkage Study (method comparison)

Calo counterpart of `celeb_study/celeb_method_comparison.ipynb`; supersedes
the original `CombinedStatisticalAnalysis_method_comparison.ipynb` and
`TestingDDNM.ipynb` with the same improvements:

- runs auto-discovered from the study directory; each truncated to its
  **completed** image count (progress.txt), non-finite images excluded
  (no hardcoded `N_EVENTS`);
- full-image ratio / Pearson / event-z via exact box decomposition
  (sample = truth outside the dead region);
- CRPS via the sorted O(J log J) estimator;
- Part B: the TestingDDNM shrinkage study for **all** algorithms.

All quantities in GeV.  Paths are env-overridable (`CALO_ROOT`).

**Analysis space:** set `SPACE = 'gev'` (default) or `'log'` below (or env
`CALO_SPACE`).  The DDPM samples in log space $x = \ln E$; stored results
are $e^{x}$, so `log` mode recovers the **raw DDPM output exactly**.
Rank-based tests (PIT, coverage) are invariant under this monotone map and
give identical results in both modes; moment-based tests (z-scores, bias,
CRPS, sharpness) and the Part-B shrinkage regression change — in log space
they are the statistically cleaner versions (near-Gaussian posteriors).
Energy-sum diagnostics (1–3) are physics observables and always use GeV.

In [ ]:
import glob, json, os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm as MplLogNorm
from scipy import stats

DATA_ROOT   = os.environ.get('CALO_ROOT', 'workdir')     # repo-root relative
STUDY_GLOB  = f'{DATA_ROOT}/inpaint_study/*_box*_eta*_phi*_S*'
EVENTS_PATH = os.environ.get(
    'CALO_EVENTS', f'{DATA_ROOT}/generated_events/events_cent0_seed0.npy')

COLORS = {'ddnm':'r', 'repaint':'g', 'ddrm':'b', 'mcg2':'y', 'pigdm2':'m',
          'mcg':'y', 'pigdm':'m'}

SPACE = os.environ.get('CALO_SPACE', 'gev')     # 'gev' or 'log' (raw DDPM)
assert SPACE in ('gev', 'log')
UNIT = '[GeV]' if SPACE == 'gev' else '[ln GeV]'
print('analysis space for value-level diagnostics:', SPACE)

## Load runs (auto-discovered, progress-truncated, NaN-guarded)

In [ ]:
true_events = np.asarray(np.load(EVENTS_PATH, mmap_mode='r'))
print('truth events:', EVENTS_PATH, true_events.shape)

runs = {}
for run_dir in sorted(glob.glob(STUDY_GLOB)):
    if not os.path.exists(f'{run_dir}/results.npy'):
        continue
    meta = json.load(open(f'{run_dir}/metadata.json'))
    alg  = meta['algorithm']

    res = np.load(f'{run_dir}/results.npy')          # (n, J, b, b)
    tru = np.load(f'{run_dir}/truth.npy')            # (n, b, b)

    done = res.shape[0]
    if os.path.exists(f'{run_dir}/progress.txt'):
        done = int(open(f'{run_dir}/progress.txt').read().split('/')[0])
    res, tru = res[:done], tru[:done]

    finite = np.isfinite(res).all(axis=tuple(range(1, res.ndim)))
    if (~finite).any():
        print(f'WARNING {alg}: {(~finite).sum()}/{done} non-finite images excluded')
    res, tru = res[finite], tru[finite]

    # unify shapes with the celeb notebook: (n, J, C=1, b, b)
    runs[alg] = {
        'inpaint_dead': res[:, :, None].astype(np.float64),
        'truth_dead'  : tru[:, None].astype(np.float64),
        'idx'         : np.where(finite)[0],
        'Y0': meta.get('eta0', meta.get('y0')),
        'X0': meta.get('phi0', meta.get('x0')),
        'BOX': meta['box'], 'meta': meta,
    }
    print(f'{alg:8s} n={finite.sum()}  J={res.shape[1]}  '
          f'box={meta["box"]}  S={meta["S"]}')

# value-level views in the chosen analysis space (log = raw DDPM output;
# exp() at storage time is exactly inverted by ln)
for r in runs.values():
    if SPACE == 'log':
        r['val_dead']  = np.log(np.clip(r['inpaint_dead'], 1e-30, None))
        r['val_truth'] = np.log(np.clip(r['truth_dead'],  1e-30, None))
    else:
        r['val_dead']  = r['inpaint_dead']
        r['val_truth'] = r['truth_dead']

algs = list(runs)
assert algs, 'no completed runs found'

## Precompute per-image quantities (exact box decomposition)

In [ ]:
for alg, r in runs.items():
    imgs = true_events[r['idx']].astype(np.float64)[:, None]   # (n,1,H,W)
    r['true_full_sum']  = imgs.sum(axis=(1, 2, 3))
    r['true_full_mean'] = imgs.mean(axis=(1, 2, 3))
    r['true_box_sum']   = r['truth_dead'].sum(axis=(1, 2, 3))
    r['reco_box_sum']   = r['inpaint_dead'].sum(axis=(2, 3, 4))
    r['reco_full_sum']  = (r['true_full_sum'][:, None]
                           - r['true_box_sum'][:, None] + r['reco_box_sum'])
    r['D']        = imgs[0].size
    r['dot_tt']   = (imgs ** 2).sum(axis=(1, 2, 3))
    r['dot_tbtb'] = (r['truth_dead'] ** 2).sum(axis=(1, 2, 3))
    r['dot_tbsb'] = (r['truth_dead'][:, None] * r['inpaint_dead']).sum(axis=(2, 3, 4))
    r['dot_sbsb'] = (r['inpaint_dead'] ** 2).sum(axis=(2, 3, 4))
    del imgs
print('precomputed.')

## Diagnostic 1 — Missing energy ratio (accuracy)

In [ ]:

def log_bins(vals, n):
    """log-spaced bins spanning the (positive) data — robust across
    centralities instead of hardcoded ranges."""
    v = np.asarray(vals); v = v[np.isfinite(v) & (v > 0)]
    return np.geomspace(v.min() * 0.8, v.max() * 1.2, n)

def binned_weighted_stats(x, y, dy, bins):
    x, y, dy, bins = map(np.asarray, (x, y, dy, bins))
    idx = np.digitize(x, bins) - 1
    nb  = len(bins) - 1
    centers = 0.5 * (bins[:-1] + bins[1:])
    mean = np.full(nb, np.nan); err = np.full(nb, np.nan); cnt = np.zeros(nb, int)
    for i in range(nb):
        m = (idx == i) & (dy > 0)
        cnt[i] = m.sum()
        if cnt[i] == 0: continue
        w = 1.0 / dy[m] ** 2
        mean[i] = np.sum(w * y[m]) / w.sum()
        err[i]  = np.sqrt(1.0 / w.sum())
    return centers, mean, err, cnt

def binned_mean_stats(x, y, bins):
    x, y, bins = map(np.asarray, (x, y, bins))
    idx = np.digitize(x, bins) - 1
    nb  = len(bins) - 1
    centers = 0.5 * (bins[:-1] + bins[1:])
    mean = np.full(nb, np.nan); err = np.full(nb, np.nan); cnt = np.zeros(nb, int)
    for i in range(nb):
        m = idx == i
        cnt[i] = m.sum()
        if cnt[i] == 0: continue
        v = y[m]
        mean[i] = v.mean()
        err[i]  = v.std(ddof=1) / np.sqrt(cnt[i]) if cnt[i] > 1 else 0.0
    return centers, mean, err, cnt

bins = log_bins(np.concatenate([r['truth_dead'].mean(axis=(1,2,3))
                                for r in runs.values()]), 26)
plt.figure()
for alg, r in runs.items():
    ratio = r['true_box_sum'][:, None] / r['reco_box_sum']
    centers, mean, sig, _ = binned_weighted_stats(
        r['truth_dead'].mean(axis=(1, 2, 3)),
        ratio.mean(axis=1), ratio.std(axis=1), bins)
    plt.errorbar(centers, mean, yerr=sig, fmt=COLORS.get(alg, 'k')+'.',
                 capsize=4, label=alg)
plt.axhline(1.0, ls='--', color='k', lw=0.8)
plt.xscale('log')
plt.xlabel('average true missing energy [GeV]')
plt.ylabel('true / reco missing energy')
plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

## Diagnostic 2 — Full-image reco/true energy ratio

In [ ]:
bins = log_bins(np.concatenate([r['true_full_mean']
                                for r in runs.values()]), 41)
plt.figure()
for alg, r in runs.items():
    ratio = r['reco_full_sum'] / r['true_full_sum'][:, None]
    centers, mean, sig, _ = binned_weighted_stats(
        r['true_full_mean'], ratio.mean(axis=1), ratio.std(axis=1) + 1e-12,
        bins)
    plt.errorbar(centers, mean, yerr=sig, fmt=COLORS.get(alg, 'k')+'.',
                 capsize=4, label=alg)
plt.axhline(1.0, ls='--', color='k', lw=0.8)
plt.xscale('log')
plt.xlabel('average true energy [GeV]')
plt.ylabel('average reco/true energy')
plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

## Diagnostic 3 — Full-image Pearson (algebraic, exact)

In [ ]:
bins = log_bins(np.concatenate([r['true_full_mean']
                                for r in runs.values()]), 41)
plt.figure()
for alg, r in runs.items():
    D = r['D']
    t_mean = r['true_full_mean']
    s_mean = r['reco_full_sum'] / D
    dot_ts = (r['dot_tt'][:, None] - r['dot_tbtb'][:, None] + r['dot_tbsb'])
    dot_ss = (r['dot_tt'][:, None] - r['dot_tbtb'][:, None] + r['dot_sbsb'])
    cov = dot_ts - D * t_mean[:, None] * s_mean
    vt  = r['dot_tt'] - D * t_mean ** 2
    vs  = dot_ss - D * s_mean ** 2
    pearson = cov / np.sqrt(np.clip(vt[:, None] * vs, 1e-30, None))
    centers, mean, sig, _ = binned_mean_stats(t_mean, pearson.mean(axis=1), bins)
    plt.errorbar(centers, mean, yerr=sig, fmt=COLORS.get(alg, 'k')+'.',
                 capsize=4, label=alg)
plt.xscale('log')
plt.xlabel('average true energy [GeV]')
plt.ylabel('average full-image Pearson coefficient')
plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

## Diagnostic 4 — z-score diagnostics (dead region)

In [ ]:
def zscore_diag(r):
    t, s = r['val_truth'], r['val_dead']
    mu = s.mean(axis=1); sd = s.std(axis=1, ddof=1)
    sd[sd < 1e-12] = np.nan
    z = (t - mu) / sd
    bm_s = s.mean(axis=(2, 3, 4))
    ev_mu, ev_sd = bm_s.mean(axis=1), bm_s.std(axis=1, ddof=1)
    ev_sd[ev_sd < 1e-12] = np.nan
    ev_z = (t.mean(axis=(1, 2, 3)) - ev_mu) / ev_sd
    return {'event_z': ev_z,
            'rms_z': np.sqrt(np.nanmean(z**2, axis=(1, 2, 3))),
            'frac_lt1': np.nanmean(np.abs(z) < 1, axis=(1, 2, 3))}

zres = {alg: zscore_diag(r) for alg, r in runs.items()}
bins = log_bins(np.concatenate([r['true_full_mean']
                                for r in runs.values()]), 21)
for key, ylabel, ref in [('event_z', 'average z-score (box mean)', 0.0),
                         ('rms_z', 'pixel RMS(z)', 1.0),
                         ('frac_lt1', 'fraction |z| < 1', 0.6827)]:
    plt.figure(figsize=(7, 4.5))
    for alg, res in zres.items():
        centers, mean, sig, _ = binned_mean_stats(
            runs[alg]['true_full_mean'], res[key], bins)
        plt.errorbar(centers, mean, yerr=sig, fmt=COLORS.get(alg, 'k')+'.',
                     capsize=4, label=alg)
    plt.axhline(ref, ls='--', color='k')
    plt.xscale('log')
    plt.xlabel('average true energy [GeV]'); plt.ylabel(ylabel)
    plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

## Diagnostic 5 — Spatial bias maps

In [ ]:
fig, axes = plt.subplots(1, len(algs), figsize=(4*len(algs), 3.4),
                         squeeze=False)
for a, alg in zip(axes[0], algs):
    r = runs[alg]
    bias = (r['val_dead'].mean(axis=1) - r['val_truth']).mean(axis=(0, 1))
    vmax = np.abs(bias).max()
    im = a.imshow(bias, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    a.set_title(alg, fontsize=10)
    a.set_xlabel('phi index'); a.set_ylabel('eta index')
    plt.colorbar(im, ax=a, fraction=0.046, label=UNIT)
fig.suptitle('spatial bias: mean(inpainted − truth) ' + UNIT)
plt.tight_layout(); plt.show()

## Diagnostic 6 — PIT histograms

PIT (like coverage below) is invariant under monotone maps — identical in gev and log modes.

In [ ]:
fig, axes = plt.subplots(1, len(algs), figsize=(3.2*len(algs), 3),
                         squeeze=False)
for a, alg in zip(axes[0], algs):
    r = runs[alg]
    pit = np.mean(r['inpaint_dead'] <= r['truth_dead'][:, None], axis=1).ravel()
    a.hist(pit, bins=20, range=(0, 1), density=True, alpha=0.7,
           color=COLORS.get(alg, 'k'))
    a.axhline(1.0, color='k', ls='--')
    a.set_title(alg, fontsize=10); a.set_xlabel('PIT')
axes[0][0].set_ylabel('density')
plt.tight_layout(); plt.show()

## Diagnostic 7 — Empirical coverage

In [ ]:
def empirical_coverage(t, s, levels=(0.5, 0.8, 0.9, 0.95)):
    out = {}
    for level in levels:
        lo = np.quantile(s, (1-level)/2, axis=1)
        hi = np.quantile(s, 1-(1-level)/2, axis=1)
        out[level] = float(((t >= lo) & (t <= hi)).mean())
    return out

print('Empirical coverage (nominal -> observed):')
for alg, r in runs.items():
    cov = empirical_coverage(r['truth_dead'], r['inpaint_dead'])
    print(f'  {alg:8s}: ' + ', '.join(f'{l:.0%}->{o:.1%}' for l, o in cov.items()))

## Diagnostic 8 — CRPS and sharpness (sorted estimator)

In [ ]:
def crps_sorted(t, s):
    J = s.shape[-1]
    term1 = np.abs(s - t[..., None]).mean(axis=-1)
    ss = np.sort(s, axis=-1)
    i  = np.arange(1, J + 1)
    term2 = (2 / (J * J)) * ((2 * i - J - 1) * ss).sum(axis=-1)
    return term1 - 0.5 * term2

crps_by, sharp_by = {}, {}
for alg, r in runs.items():
    s = np.moveaxis(r['val_dead'], 1, -1)
    crps_by[alg]  = float(crps_sorted(r['val_truth'], s).mean())
    sharp_by[alg] = float(r['val_dead'].std(axis=1).mean())

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
a1.bar(list(crps_by), list(crps_by.values()),
       color=[COLORS.get(a, 'k') for a in crps_by])
a1.set_ylabel('mean CRPS ' + UNIT); a1.set_title('CRPS (lower = better)')
a2.bar(list(sharp_by), list(sharp_by.values()),
       color=[COLORS.get(a, 'k') for a in sharp_by])
a2.set_ylabel('mean ensemble std ' + UNIT); a2.set_title('sharpness')
plt.tight_layout(); plt.show()
for alg in crps_by:
    print(f'  {alg:8s}: CRPS={crps_by[alg]:.5f}  sharpness={sharp_by[alg]:.5f}')

## Diagnostic 9 — Example reconstructions

Intensity panels on a log color scale (the original used linear, which
washes out against the 1e-3 GeV floor).

In [ ]:
EVENT_IDX = 0
for alg, r in runs.items():
    i_img = r['idx'][EVENT_IDX]
    Y0, X0, B = r['Y0'], r['X0'], r['BOX']
    truth = true_events[i_img].astype(np.float64)
    mean_img = truth.copy()
    mean_img[Y0:Y0+B, X0:X0+B] = r['inpaint_dead'][EVENT_IDX].mean(axis=0)[0]
    one_img = truth.copy()
    one_img[Y0:Y0+B, X0:X0+B] = r['inpaint_dead'][EVENT_IDX, 0, 0]
    resid = mean_img - truth

    fig, ax = plt.subplots(1, 4, figsize=(16, 3.2))
    for a, (img, ttl) in zip(ax, [(truth, 'truth'), (mean_img, 'ensemble mean'),
                                  (one_img, 'one sample'), (resid, 'mean − truth')]):
        if ttl == 'mean − truth':
            v = np.abs(img).max() + 1e-9
            im = a.imshow(img, cmap='RdBu_r', vmin=-v, vmax=v, aspect='auto')
        else:
            im = a.imshow(np.clip(img, 1e-3, None),
                          norm=MplLogNorm(1e-3, max(img.max(), 1e-2)),
                          cmap='viridis', aspect='auto')
        a.add_patch(plt.Rectangle((X0-.5, Y0-.5), B, B, fill=False,
                                  edgecolor='lime', lw=1.5))
        a.set_title(ttl, fontsize=10)
        plt.colorbar(im, ax=a, fraction=0.046)
    fig.suptitle(f'{alg} — event {i_img} (green = inpainted region)')
    plt.tight_layout(); plt.show()

# Part B — Shrinkage study, all algorithms

TestingDDNM generalized: per-image dead-box mean of truth vs posterior
mean (GeV), regression fit per algorithm, 2D-histogram view, summary
table.  Slope < 1 is expected posterior-mean shrinkage; cross-algorithm
(and calo-vs-celeb) comparison is the point.

In [ ]:
fig, axes = plt.subplots(2, len(algs), figsize=(4.2*len(algs), 8),
                         squeeze=False)
summary = {}
for col, alg in enumerate(algs):
    r = runs[alg]
    t_m = r['val_truth'].mean(axis=(1, 2, 3))
    f_m = r['val_dead'].mean(axis=(1, 2, 3, 4))
    f_s = r['val_dead'].mean(axis=(2, 3, 4)).std(axis=1, ddof=1)

    res_f = stats.linregress(t_m, f_m)      # forward: fill vs truth
    res_r = stats.linregress(f_m, t_m)      # reverse: truth vs fill
    summary[alg] = (res_f, res_r)
    lim = [min(t_m.min(), f_m.min()), max(t_m.max(), f_m.max())]

    a = axes[0][col]                        # forward (shrinkage view)
    a.errorbar(t_m, f_m, yerr=f_s, fmt='o', ms=3, alpha=0.5, capsize=2,
               color=COLORS.get(alg, 'k'))
    xx = np.linspace(t_m.min(), t_m.max(), 50)
    a.plot(xx, res_f.intercept + res_f.slope*xx, 'k-', lw=2,
           label=f'y={res_f.slope:.2f}x+{res_f.intercept:.2f}\nR²={res_f.rvalue**2:.2f}')
    a.plot(lim, lim, 'r--', lw=1, label='y = x')
    a.set_title(alg); a.set_xlabel('truth box mean ' + UNIT)
    a.grid(alpha=0.3); a.legend(fontsize=8)

    a = axes[1][col]                        # reverse (calibration view)
    a.errorbar(f_m, t_m, xerr=f_s, fmt='s', ms=3, alpha=0.5, capsize=2,
               color=COLORS.get(alg, 'k'))
    xx = np.linspace(f_m.min(), f_m.max(), 50)
    a.plot(xx, res_r.intercept + res_r.slope*xx, 'k-', lw=2,
           label=f'y={res_r.slope:.2f}x+{res_r.intercept:.2f}\nR²={res_r.rvalue**2:.2f}')
    a.plot(lim, lim, 'r--', lw=1, label='y = x')
    a.set_xlabel('posterior-mean box mean ' + UNIT)
    a.grid(alpha=0.3); a.legend(fontsize=8)

axes[0][0].set_ylabel('posterior-mean box mean ' + UNIT)
axes[1][0].set_ylabel('truth box mean ' + UNIT)
plt.tight_layout(); plt.show()

print('Shrinkage summary (box means, ' + SPACE + '):')
print('  forward slope: fill~truth (shrinkage, <1 expected);')
print('  reverse slope: truth~fill (calibration: ~1 for an unbiased')
print('  posterior mean); identity check: fwd*rev = R^2')
for alg, (rf, rr) in summary.items():
    print(f'  {alg:8s}: fwd={rf.slope:.3f}  rev={rr.slope:.3f}  '
          f'R²={rf.rvalue**2:.3f}  (fwd*rev={rf.slope*rr.slope:.3f})')

In [ ]:
fig, axes = plt.subplots(1, len(algs), figsize=(4*len(algs), 3.6),
                         squeeze=False)
for a, alg in zip(axes[0], algs):
    r = runs[alg]
    t_m = r['val_truth'].mean(axis=(1, 2, 3))
    f_m = r['val_dead'].mean(axis=(1, 2, 3, 4))
    h = a.hist2d(t_m, f_m, bins=15, cmap='viridis')
    lim = [min(t_m.min(), f_m.min()), max(t_m.max(), f_m.max())]
    a.plot(lim, lim, 'r--', lw=1)
    a.set_title(alg, fontsize=10); a.set_xlabel('truth box mean ' + UNIT)
    plt.colorbar(h[3], ax=a, fraction=0.046)
axes[0][0].set_ylabel('posterior-mean box mean ' + UNIT)
plt.tight_layout(); plt.show()